# Ablation Study — AugCRNN-T (TAM VERİ)

> **Önemli:** Önceki tüm sayılar (78.06 / 84.54, N=5,338) IAM'in **%39'luk kesik**
> bir etiket dosyasıyla elde edilmişti. Repo artık tam IAM etiketlerinden kurulmuş
> split dosyalarını taşıyor (`aachen_splits/*_words.txt`, resmi 747/116/336 form).
> Bu yüzden **her şey sıfırdan, tam veriyle** ölçülür. Kaggle'daki `words.txt`
> artık kullanılmıyor; sadece görüntüler kullanılıyor.

| Split | Form | Kelime (ok) |
|---|---:|---:|
| train | 747 | 47,999 |
| validation | 116 | 7,559 |
| test | 336 | **20,310** |

Tam veri ~1.5× büyük, bir eğitim ~3–3.5 saat. Kaggle oturumu 12 saat olduğu için
**iki oturum** gerekiyor. `SESSION` değişkeniyle seçilir.

## Oturum A — ana sayılar (2 eğitim, ~7 saat)
| `--aug-mode` | Anlamı |
|---|---|
| `narrow` | **CRNN-L baseline** (dar fotometrik, elastik ✗, morf ✗) |
| `full` | **AugCRNN-T** (önerilen) |

Ardından Tablo B (lexicon ablation) `full` modelinin üstünde çalışır (~20 dk).

## Oturum B — bileşen ablation'ı (3 eğitim, ~10 saat)
| `--aug-mode` | Anlamı |
|---|---|
| `photo` | geniş fotometrik, elastik ✗, morf ✗ |
| `elastic` | geniş fotometrik + elastik |
| `morph` | geniş fotometrik + morfolojik |

## Gerekli Input
- IAM word dataset (`words/` görüntü klasörü olan herhangi biri; `words.txt`'si kesik olsa da fark etmez)

## Settings
Accelerator **GPU T4**, Internet **ON**, **Save & Run All (Commit)**


In [ ]:
# Hücre 1: oturum seçimi + ortam + repo
SESSION = "A"          # "A" = narrow + full + lexicon ablation   |   "B" = photo + elastic + morph

import torch, sys, os, subprocess, shutil, json
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK'}  |  PyTorch {torch.__version__}  |  SESSION {SESSION}")

REPO_URL = "https://github.com/Ridvan013/CRNN-Handwriting-Recognition.git"
BRANCH   = "feature/aachen-v3-extended-trigram"
REPO_DIR = "/kaggle/working/repo"
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,REPO_DIR], check=True)
for name in ["cloud","aachen_splits","trigram_lm.py","verify_aachen_splits.py"]:
    src, dst = os.path.join(REPO_DIR,name), os.path.join("/kaggle/working",name)
    shutil.copytree(src,dst,dirs_exist_ok=True) if os.path.isdir(src) else shutil.copy(src,dst)
sys.path.insert(0,"/kaggle/working"); os.chdir("/kaggle/working")

# split dosyalari TAM veriden mi? (kesik dosyayla 5,338 olurdu)
n_test = sum(1 for l in open("aachen_splits/test_words.txt") if l.strip() and not l.startswith("#"))
assert n_test == 20310, f"test_words.txt {n_test} satir — repo eski! (20,310 bekleniyor)"
print(f"split dosyalari tam veri: test={n_test:,}")

import nltk
try: nltk.data.find("corpora/words")
except LookupError: nltk.download("words", quiet=True)


In [ ]:
# Hücre 2: IAM görüntü klasörü (words.txt'ye artık ihtiyaç yok)
import subprocess, os
res = subprocess.run(["find","/kaggle/input","-maxdepth","6","-type","d","-name","words"], capture_output=True, text=True)
IAM_ROOT = None
for d in [p.strip() for p in res.stdout.splitlines() if p.strip()]:
    if os.path.isdir(os.path.join(d,"a01")): IAM_ROOT = d; break
assert IAM_ROOT, "IAM words/ görüntü klasörü bulunamadı"
print("IAM words/ :", IAM_ROOT)
n_png = sum(len(f) for _,_,f in os.walk(IAM_ROOT)); print(f"png sayısı  : {n_png:,}  (115,320 bekleniyor)")


---
## Split doğrulaması

Eğitime başlamadan önce bölmenin bütünlüğü kontrol edilir: form/yazar/metin ayrıklığı, resmi listeyle eşleşme, kayıt ve görüntü bütünlüğü. Bir kontrol bile geçmezse hücre hata verir ve eğitim başlamaz.

In [ ]:
subprocess.run(["python","verify_aachen_splits.py","--img-root",IAM_ROOT], check=True)

---
## Eğitimler
Seçilen oturumun modları sırayla eğitilir. Sadece `--aug-mode` değişir; epoch/batch/lr/patience/seed sabit.

In [ ]:
MODES = {"A": ["narrow", "full"], "B": ["photo", "elastic", "morph"]}[SESSION]
for mode in MODES:
    print("
" + "="*70 + f"
  --aug-mode {mode}
" + "="*70)
    subprocess.run(["python","cloud/v3_augmented_train.py","--aug-mode",mode,
                    "--epochs","100","--batch","128","--lr","7e-4","--patience","15",
                    "--model-dir",f"/kaggle/working/abl_{mode}",
                    "--iam-root",IAM_ROOT], check=True)


---
## Tablo B — Lexicon / trigram ablation (yalnız Oturum A)
`full` modelinin greedy çıktısına dört post-processing uygulanır; eğitim yok.

In [ ]:
if SESSION == "A":
    subprocess.run(["python","cloud/ablation_lexicon.py",
                    "--model","/kaggle/working/abl_full/best_model_wa.pth",
                    "--iam-root",IAM_ROOT,
                    "--out","/kaggle/working/results/ablation_lexicon.json"], check=True)
else:
    print("Oturum B: lexicon ablation atlandı (Oturum A'da yapılır)")


---
## Özet tablo

In [ ]:
import json, os, csv, math
def wa_cer(path):
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    k = sum(1 for r in rows if str(r.get("correct", r.get("Is_Correct",""))).strip().lower() in ("1","true"))
    n = len(rows); p=k/n; z=1.96
    lo=(p+z*z/(2*n)-z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)))/(1+z*z/n); hi=(p+z*z/(2*n)+z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)))/(1+z*z/n)
    return k, n, p*100, lo*100, hi*100

LABEL = {"narrow":"CRNN-L (baseline)", "photo":"+ wide photometric", "elastic":"+ elastic",
         "morph":"+ morphological", "full":"AugCRNN-T (proposed)"}
print("="*78); print(" TABLO A — Augmentation ablation  (tam Aachen test, N=20,310)"); print("="*78)
print(f"{'Configuration':<26s}{'WA (%)':>9s}{'95% CI':>18s}{'k/n':>16s}")
for mode in ["narrow","photo","elastic","morph","full"]:
    c = f"/kaggle/working/abl_{mode}/test_results_analysis.csv"
    if os.path.exists(c):
        k,n,wa,lo,hi = wa_cer(c); print(f"{LABEL[mode]:<26s}{wa:>9.2f}{f'[{lo:.2f}, {hi:.2f}]':>18s}{f'{k}/{n}':>16s}")
    else:
        print(f"{LABEL[mode]:<26s}{'—':>9s}{'(bu oturumda yok)':>18s}")

p = "/kaggle/working/results/ablation_lexicon.json"
if os.path.exists(p):
    r = json.load(open(p))
    print("
"+"="*78); print(" TABLO B — Lexicon / trigram ablation"); print("="*78)
    print(f"{'Configuration':<42s}{'WA (%)':>9s}{'CER (%)':>9s}")
    for c in r["configurations"]: print(f"{c['name']:<42s}{c['wa_pct']:>9.2f}{c['cer_pct']:>9.2f}")
    print(f"
Lexicon boyutları: {r['lexicon_sizes']}")
